In [ ]:
# !pwd
!pip install -e ..

In [ ]:
from blockhouse_ml.options.utils.data_handler import DataProcessor
from blockhouse_ml.options.utils.fetch_merge_data import PolygonClient

In [ ]:
import os
import pandas as pd
import requests
import numpy as np

In [ ]:
DATA_DIR = "../TempData1"
os.makedirs(DATA_DIR, exist_ok=True)

data_client = PolygonClient(DATA_DIR)

data_processor = DataProcessor()

# start_date = "2024-08-23"
# end_date = "2024-07-25"

start_date = '2024-08-18'
end_date = '2024-08-19'

In [ ]:
# filepath = f"{DATA_DIR}/fetched_option_AAPL240823C00100000_{start_date}_{end_date}.csv"
# if not os.path.exists(filepath):
data  = data_client.fetch_and_merge_data("AAPL",start_date,end_date,'240823','C',100)
data.to_csv(f"{DATA_DIR}/fetched_option_AAPL240823C00100000.csv")
# else:
    # data = pd.read_csv(filepath, index_col='timestamp', parse_dates=True)

In [ ]:
data

In [ ]:
import requests
contract = data_client.get_contract_name("AAPL","240820","C",100)
print(contract)
contract="asd"
url = f"https://api.polygon.io/v3/reference/options/contracts/{contract}?as_of=2024-08-20&apiKey={data_client.api_key}"
response = requests.get(url)
data = response.json()
print(data)

In [ ]:
data_processed = data_processor.process_data(data,option_type="C",strike_price=100,n_jobs=2)
data_processed

In [ ]:
data_processed.to_csv(f"{DATA_DIR}/processed_option_AAPL240823C00100000.csv")

In [ ]:
print(data_processed['transaction_cost'].describe())
print(data_processed['volume'].describe())
print(data_processed['forecast_3Hr_volume'].describe())

In [ ]:
data_processed.info()

In [ ]:
import requests
import pandas as pd
from time import sleep

api_key = "r65B9O5aplSJn7BWSo8z8pNH8v2wW5yc"
def get_all_quotes(ticker, start_time, end_time):
    quotes = []
    next_url = f'https://api.polygon.io/v3/quotes/{ticker}'
    params = {
        "apiKey": api_key,
        "timestamp.gte": start_time.value,
        "timestamp.lt": end_time.value,
        "limit": 50000,
        "order": "asc",
        "sort": "timestamp"
    }

    while next_url:
        response = requests.get(next_url, params=params)
        response.raise_for_status()
        data = response.json()
        quotes.extend(data['results'])
        
        next_url = data.get('next_url')
        if next_url:
            print(f"Next URL: {next_url}")
        #     params = {"apiKey": api_key}  # Reset params as next_url includes other parameters
        
        # Respect rate limits
        sleep(0.2)  # Wait 200ms between requests to avoid hitting rate limits

    return quotes
def make_options_dataframe(contract:str, start_date:str, end_date:str):
    ticker = 'O:'+contract
    # List Quotes
    # quotes = []
    # quote = self.client.list_quotes(ticker=ticker, timestamp_gte = start_date, timestamp_lt= end_date, sort='timestamp')
    # for value in quote:
    #     quotes.append(value)
    all_quotes = []

    for date in pd.date_range(start=start_date, end=pd.to_datetime(end_date), freq='D').strftime('%Y-%m-%d').tolist():
        day = pd.to_datetime(date)

        # Set start after 13:00
        start_time = day + pd.Timedelta(hours=13)

        # Loop through each hour
        for n in range(8):  # 8 hours from 13:00 to 21:00
            hour_start = start_time + pd.Timedelta(hours=n)
            hour_end = hour_start + pd.Timedelta(hours=1)
            
            quotes = get_all_quotes(ticker, hour_start, hour_end)
            all_quotes.extend(quotes)

    # Create DataFrame
    df_quotes = pd.DataFrame(all_quotes)
    # print(df_quotes.columns)
    # df_quotes.reset_index(drop=True, inplace=True)
    # df_quotes['timestamp'] = pd.to_datetime(df_quotes['sip_timestamp'], unit='ns', utc=True)
    # # Convert UTC to Eastern Time (ET)
    # df_quotes['timestamp'] = df_quotes['timestamp'].dt.tz_convert('US/Eastern')
    # # Remove timezone information and keep as datetime with only date, hour, and minute
    # df_quotes['timestamp'] = df_quotes['timestamp'].dt.floor('min').dt.tz_localize(None)
    # df_quotes = df_quotes[['timestamp', 'ask_price', 'ask_size', 'bid_price', 'bid_size']]
    return df_quotes
def resample_to_daily(data):
    data.set_index('timestamp', inplace=True)
    daily_data = data.resample('min').agg({
        'ask_price': 'mean',
        'ask_size': 'sum',
        'bid_price': 'mean',
        'bid_size': 'sum'
    }).dropna()
    return daily_data


In [ ]:
contract = 'AAPL240823C00100000'
start_date = "2024-07-23"
end_date = "2024-07-25"
print(contract)
# df = make_options_dataframe(contract, start_date, end_date)
# df = resample_to_daily(df)

In [ ]:
# df

In [ ]:
 # Convert timestamp to nanoseconds

from polygon import RESTClient
import pytz
import requests
from time import sleep
import pandas as pd
from datetime import datetime, timezone

data_dir = "../OptionsData"
quicker_training_filepath = f'{data_dir}/options-train-data.csv'
ticker= 'AAPL'
## User specific Option data
user_data = {
    "option_type" : "C",
    "strike_price" : 100,
    "maturity_date" : "240820"
}
data = pd.read_csv(quicker_training_filepath)
# Ensure the 'datetime' column is in datetime format
data['datetime'] = pd.to_datetime(data['datetime'])

data.set_index('datetime', inplace=True)
# print(data)
data = data.between_time('13:30', '20:00')

data.reset_index(inplace=True)
current_step = 0
est_tz = pytz.timezone('America/New_York')


current_timestamp = data['datetime'].iloc[current_step]
# timestamp = pd.to_datetime(current_timestamp).replace(tzinfo=pytz.UTC)


timestamp = pd.Timestamp("2024-07-05 14:00:00+00:00")
api_key = "r65B9O5aplSJn7BWSo8z8pNH8v2wW5yc"
contract = "O:AAPL240823C00100000"
# timestamp = pd.to_datetime("2024-07-23 13:00:00")

# timestamp = pd.Timestamp("2024-07-05 14:09:00")
client = RESTClient(api_key)
# start_timestamp = int(timestamp.timestamp() * 1_000_000_000)
# end_timestamp = int(start_timestamp + (120 * 1_000_000_000))   # Subtract 1 minute from the timestamp for start time

# start_date = pd.to_datetime("2024-07-10 10:00:00",)
# end_date = pd.Timestamp("2024-07-10 10:10:00")
# start_timestamp = 1721016720000000000
# end_timestamp = 1721016780000000000
# start_timestamp = int(.timestamp() * 1_000_000_000)
# # start_timestamp = int(pd.to_datetime(" 2024-07-05 14:40:00").timestamp() * 1_000_000_000)
# end_timestamp = int(pd.to_datetime("2024-07-10 10:10:00").timestamp() * 1_000_000_000)
start_timestamp = 1720529400000000000
end_timestamp = 1720529460000000000
print(pd.to_datetime(start_timestamp, unit='ns').tz_localize(est_tz), pd.to_datetime(end_timestamp, unit='ns').tz_localize(est_tz))

# next_url = f'https://api.polygon.io/v3/quotes/{contract}'
# params = {
#     "apiKey": api_key,
#     "timestamp.gte": start_timestamp,
#     "timestamp.lte": end_timestamp,
#     "limit": 50000,
#     "order": "asc",
#     "sort": "timestamp"
# }
# quotes = []

# while next_url:
#     response = requests.get(next_url, params=params)
#     response.raise_for_status()
#     data = response.json()
#     print(data)
#     quotes.extend(data['results'])
    
#     next_url = data.get('next_url')
#     # if next_url:
#     #     params = {"apiKey": self.api_key}  # Reset params as next_url includes other parameters
    
#     # Respect rate limits
#     sleep(0.2)  # Wait 200ms between requests to avoid hitting rate limits


# Make the API call
quotes = client.list_quotes(
    contract,
    timestamp_gte=start_timestamp,
    timestamp_lte=end_timestamp,
    limit=50000  # Adjust based on your needs
)
# print(start_timestamp, end_timestamp, len(quotes))

# Initialize variables to store maximum bid price and sizes
max_bid_price = float('-inf')
bid_sizes = []
ask_sizes = []
bid_prices = []

# Process the results to find the maximum bid price and collect sizes
for quote in quotes:
    print(quote)
    if quote.bid_price > max_bid_price:
        max_bid_price = quote.bid_price
    bid_prices.append(quote.bid_price)
    bid_sizes.append(quote.bid_size)
    ask_sizes.append(quote.ask_size)
api_keys_databento = ""
client = db.Historical(api_keys_databento)

data = client.timeseries.get_range(
    dataset="OPRA.PILLAR", # for stocks which require MBO
    schema="mbp-1",
    stype_in="raw_symbol",
    symbols=["AAPL  240823C00100000"],
    start=start_timestamp,
    end=end_timestamp,
)
df = data.to_df()
df

In [ ]:
max_bid_price

In [ ]:
import databento as db
import pandas as pd
import numpy as np
import pytz

In [ ]:
start_date = '2024-07-08'+"T09:48:00"
end_date = '2024-07-08'+"T09:49:00"

In [ ]:
est_tz = pytz.timezone('America/New_York')
print(int(pd.to_datetime("2024-07-08 09:48:00").replace(tzinfo=est_tz).timestamp()*1_000_000_000))
print(int(pd.to_datetime("2024-07-08 09:48:00").timestamp()*1_000_000_000))

In [ ]:
api_keys_databento = "db-eKU7cAt4iTryxUbycEY7REuXXkwcU"
client = db.Historical(api_keys_databento)

print(pd.to_datetime("2024-07-08 09:48:00").timestamp())
start_timestamp = 1720432020000000000
end_timestamp = 1720432080000000000
contract = "AAPL  240823C00100000"

print(pd.to_datetime(start_timestamp, unit='ns'), pd.to_datetime(end_timestamp, unit='ns'))

data = client.timeseries.get_range(
    dataset="OPRA.PILLAR", # for stocks which require MBO
    schema="mbp-1",
    stype_in="raw_symbol",
    symbols=[contract],
    # start=start_timestamp,
    # end=end_timestamp,
    start=start_date,
    end=end_date,
)
df = data.to_df()

In [ ]:
df

In [ ]:
import databento as db
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
from pandas.tseries.holiday import USFederalHolidayCalendar
import pytz
def get_next_valid_market_timestamp(current_timestamp, time_slice_minutes):
    """
    Calculates the next valid market timestamp based on the current timestamp and time slice.

    Args:
    - current_timestamp (datetime, pd.Timestamp, or numpy.int64): Current market timestamp. ( Assuming the timestamp is in UTC )
    - time_slice_minutes (int): Minutes to add to the current timestamp.

    Returns:
    - datetime: The next valid market timestamp considering market hours, weekends, and holidays.
    """
    # Convert numpy.int64 (or any int) to datetime
    if isinstance(current_timestamp, (np.int64, int)):
        current_timestamp = datetime.utcfromtimestamp(current_timestamp)

    # Ensure current_timestamp is a single timestamp, not a DataFrame or Series
    if isinstance(current_timestamp, (pd.Series, pd.DataFrame)):
        current_timestamp = current_timestamp.squeeze()  # Convert to scalar if it's a Series with one element

    # Convert to datetime if it's a pandas Timestamp
    if isinstance(current_timestamp, pd.Timestamp):
        current_timestamp = current_timestamp.to_pydatetime()

    # Now it's safe to use replace
    market_start = current_timestamp.replace(hour=9, minute=30, second=0, microsecond=0)
    market_end = current_timestamp.replace(hour=16, minute=0, second=0, microsecond=0)

    # Add time slice to the current timestamp
    new_timestamp = current_timestamp + timedelta(minutes=time_slice_minutes)

    # If the new timestamp is beyond market hours
    if new_timestamp > market_end:
        # Move to the next market day's open
        new_timestamp = market_start + timedelta(days=1)

    # If the new timestamp is before market open, set it to the market start time
    if new_timestamp < market_start:
        new_timestamp = market_start

    # Skip weekends
    while new_timestamp.weekday() >= 5:  # 5 = Saturday, 6 = Sunday
        new_timestamp += timedelta(days=1)

    # Check if the new timestamp falls on a holiday
    cal = USFederalHolidayCalendar()
    holidays = cal.holidays(start=new_timestamp, end=new_timestamp + timedelta(days=365)).to_pydatetime()
    while new_timestamp in holidays:
        new_timestamp += timedelta(days=1)
        new_timestamp = new_timestamp.replace(hour=9, minute=30, second=0, microsecond=0)  # Reset to market open time

    return new_timestamp

In [ ]:
# val = pd.to_datetime(1720432020000000000, unit='ns')
val = pd.to_datetime("2024-07-05 09:30:00")
dat = get_next_valid_market_timestamp(val, 1)
dat

In [ ]:
d = pd.to_datetime(dat)

In [ ]:
quicker_training_filepath = f'../OptionsData/options-train-data-old.csv'
output_path = f'../OptionsData/macro-trader-training-data.csv'
ticker= 'AAPL'
## User specific Option data
user_data = {
    "option_type" : "C",
    "strike_price" : 100,
    "maturity_date" : "240823"
}
processed_data = pd.read_csv(quicker_training_filepath)

In [ ]:
processed_data[processed_data['datetime'] == d]

In [ ]:
processed_data

In [ ]:
import math
math.isnan(processed_data['bid_price'].iloc[0:0].mean())

In [ ]:
import pytz
from blockhouse_ml.options.utils.data_handler import DataProcessor
from blockhouse_ml.options.utils.fetch_merge_data import PolygonClient
import databento as db

data_dir = '../OptionsData'
ticker= 'AAPL'
## User specific Option data
user_data = {
    "option_type" : "C",
    "strike_price" : 100,
    "maturity_date" : "240823"
}

data_processor = DataProcessor()
data_client = PolygonClient(save_dir=data_dir)


In [ ]:

# quicker_training_filepath = f'{data_dir}/options-train-data.csv'
# output_path = f'{data_dir}/macro-trader-training-data.csv'


# est_tz = pytz.timezone('America/New_York')

# processed_data = pd.read_csv(quicker_training_filepath)
# processed_data['contract'] = f"{ticker}  {user_data['maturity_date']}C00100000"
# processed_data['datetime'] = pd.to_datetime(processed_data['timestamp'])
# # processed_data['timestamp'] = processed_data['datetime']
# processed_data['VWAP'] = processed_data['VWAP_bid']
# processed_data = processed_data[processed_data['datetime'].dt.dayofweek < 5]
# processed_data.set_index('datetime', inplace=True)
# # print(processed_data)
# processed_data = processed_data.between_time('13:30', '20:00')

# processed_data.reset_index(inplace=True)
# processed_data =data_processor.process_data(processed_data,option_type=user_data['option_type'],strike_price=user_data['strike_price'],n_jobs=2)
# # processed_data['datetime'] = processed_data['datetime']
# print(processed_data.columns)
# processed_data.to_csv(output_path)
# processed_data

In [ ]:
start_data = "2024-07-05"
end_data = "2024-07-05"

data = data_client.fetch_and_merge_data(ticker, start_data, end_data, **user_data)

In [ ]:
import pandas as pd
import time
from datetime import datetime, timedelta
from tqdm import tqdm

start_date = "2024-07-05"
end_date = "2024-07-07"

# for date in tqdm(pd.date_range(start=start_date, end=pd.to_datetime(end_date), freq='D').strftime('%Y-%m-%d').tolist()):
#     day = pd.to_datetime(date)
#     print(day)
#     start_time = day + pd.Timedelta(hours=13)
#     print(start_time.timestamp())

In [ ]:
import databento as db
client = db.Historical("db-eKU7cAt4iTryxUbycEY7REuXXkwcU")

contract = f"{ticker}  {user_data['maturity_date']}C00100000"
details = client.batch.submit_job(
    dataset="OPRA.PILLAR",
    symbols=[contract],
    stype_in="raw_symbol",
    schema="mbp-1",
    encoding="csv",
    pretty_px=True,
    split_duration="day",
    start=f"{start_date}T13:00:00+00:00",
    end=f"{end_date}T20:00:00+00:00",
)
print(details)

In [ ]:
!pip install zstandard

In [ ]:
# Once the job is finished, you can download the results from databento.com download center
import pandas as pd
import zstandard as zstd

# Path to your .zst compressed CSV file
file_path = '../OptionsData1/AAPL/opra-pillar-20240708-20240822.mbp-1.csv.zst'

# Open and decompress the file
with open(file_path, 'rb') as compressed_file:
    dctx = zstd.ZstdDecompressor()
    with dctx.stream_reader(compressed_file) as reader:
        # Read the decompressed data into a pandas DataFrame
        df = pd.read_csv(reader)

In [ ]:
df

In [ ]:
df

In [ ]:
# Once the job is finished, you can download the results from databento.com download center
import pandas as pd
import zstandard as zstd

# Path to your .zst compressed CSV file
file_path = '../OptionsData/opra-pillar-20240705.mbp-1.csv.zst'

# Open and decompress the file
with open(file_path, 'rb') as compressed_file:
    dctx = zstd.ZstdDecompressor()
    with dctx.stream_reader(compressed_file) as reader:
        # Read the decompressed data into a pandas DataFrame
        df = pd.read_csv(reader)
        df.dropna(inplace=True)
        df["datetime"] = pd.to_datetime(df['ts_recv'], unit='ns')
df

In [ ]:
def calculate_actual_price(bid_prices, bid_sizes):
    return (bid_prices * bid_sizes).sum() / bid_sizes.sum()

def calculate_spread_cost(bid_prices, ask_prices):
    return ask_prices.mean() - bid_prices.mean()

def resample_to_daily(data):
    # If you need to perform more complex calculations, you can use apply
    # Define a custom function to calculate spread cost
    def calculate_costs(group):
        # Calculate the average bid price and ask price for the minute
        bid_price = group['bid_px_00']
        ask_price = group['ask_px_00']
        ask_size = group['ask_sz_00']
        bid_size = group['bid_sz_00']

        print(len(bid_price), len(ask_price), len(bid_size), len(ask_size))

        
        # Calculate the spread cost (difference between ask and bid prices)
        spread_cost = calculate_spread_cost(bid_price, ask_price)
        actual_price = calculate_actual_price(bid_price,bid_sizes=bid_size)
        expected_price = bid_price.max()

        data = {}
        # Return a Series with the calculated spread cost
        data['spread_cost'] = spread_cost
        data['actual_price'] = actual_price
        data['expected_price'] = expected_price
        data['ask_price'] = ask_price.mean()
        data['bid_price'] = bid_price.mean()
        data['ask_size'] = ask_size.sum()
        data['bid_size'] = bid_size.sum()
        return pd.Series(data)
    
    # data.set_index('timestamp', inplace=True)
    # Resample the data by minute and apply the custom calculation
    result = data.resample('T').apply(calculate_costs)
    return result


In [ ]:
# df.set_index('datetime', inplace=True)
data = resample_to_daily(df)

In [ ]:
data

In [ ]:
from blockhouse_ml.options.utils.fetch_merge_data import DataClient

data_dir = "../OptionsData"
data_client = DataClient(data_dir)

In [ ]:
start_date = "2024-07-05"
end_date = "2024-08-10"
file_path = ['../OptionsData/opra-pillar-20240801-20240810.mbp-1.csv.zst','../OptionsData/opra-pillar-20240705-20240731.mbp-1.csv.zst'] #'../OptionsData/opra-pillar-20240705.mbp-1.csv.zst'
user_data = {
    "option_type" : "C",
    "strike_price" : 100,
    "maturity_date" : "240823"
}
data = data_client.fetch_and_merge_data(ticker="AAPL",start_date=start_date, end_date=end_date, quotes_paths=file_path, **user_data)

In [ ]:
data